In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from catboost import CatBoostRegressor

from src.config import (
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
    SUBMISSION_DIR,
    TARGET,
    ID_COL,
    RANDOM_STATE,
    N_SPLITS
)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\pc\Desktop\yzta-2026-datathon


In [14]:
def rmse(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return np.sqrt(mse)

In [15]:
def add_features(df):
    df = df.copy()

    # Uyku kalitesi
    if {"rem_yuzdesi", "derin_uyku_yuzdesi"}.issubset(df.columns):
        df["toplam_kaliteli_uyku_yuzdesi"] = (
            df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]
        )

        df["rem_derin_uyku_carpim"] = (
            df["rem_yuzdesi"] * df["derin_uyku_yuzdesi"]
        )

    # Uyku bölünmesi
    if {"gecelik_uyanma_sayisi", "uykuya_dalma_suresi_dk"}.issubset(df.columns):
        df["uyku_bolunme_yuku"] = (
            df["gecelik_uyanma_sayisi"] * df["uykuya_dalma_suresi_dk"]
        )

        df["uyku_verimsizlik_skoru"] = (
            df["uykuya_dalma_suresi_dk"] + 10 * df["gecelik_uyanma_sayisi"]
        )

    # Stres ve çalışma yükü
    if {"stres_skoru", "gunluk_calisma_saati"}.issubset(df.columns):
        df["stres_calisma_yuku"] = (
            df["stres_skoru"] * df["gunluk_calisma_saati"]
        )

    # Ekran + kafein yükü
    if {"uyku_oncesi_ekran_suresi_dk", "uyku_oncesi_kafein_mg"}.issubset(df.columns):
        df["ekran_kafein_yuku"] = (
            df["uyku_oncesi_ekran_suresi_dk"] + df["uyku_oncesi_kafein_mg"]
        )

    # Adım sayısı ölçekleme
    if "gunluk_adim_sayisi" in df.columns:
        df["adim_sayisi_bin"] = df["gunluk_adim_sayisi"] / 1000

    # Nabız + stres yükü
    if {"dinlenik_nabiz_bpm", "stres_skoru"}.issubset(df.columns):
        df["nabiz_stres_yuku"] = (
            df["dinlenik_nabiz_bpm"] * df["stres_skoru"]
        )

    # BMI kategorisi
    if "vucut_kitle_indeksi" in df.columns:
        df["bmi_kategori"] = pd.cut(
            df["vucut_kitle_indeksi"],
            bins=[0, 18.5, 25, 30, np.inf],
            labels=["zayif", "normal", "kilolu", "obez"]
        ).astype("object")

    # Hafta sonu flag
    if "gun_tipi" in df.columns:
        df["hafta_sonu_flag"] = (df["gun_tipi"] == "Hafta sonu").astype(int)

    # Ruh sağlığı risk skoru
    if "ruh_sagligi_durumu" in df.columns:
        risk_map = {
            "Saglikli": 0,
            "Anksiyete": 1,
            "Depresyon": 2,
            "Anksiyete ve depresyon": 3,
        }

        df["ruh_sagligi_risk_skoru"] = df["ruh_sagligi_durumu"].map(risk_map)

    return df

In [16]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

Train shape: (56000, 24)
Test shape: (24000, 23)
Sample submission shape: (2, 2)


,id,yas,cinsiyet,meslek,vucut_kitle_indeksi,ulke,rem_yuzdesi,derin_uyku_yuzdesi,uykuya_dalma_suresi_dk,gecelik_uyanma_sayisi,...,stres_skoru,gunluk_calisma_saati,kronotip,ruh_sagligi_durumu,dinlenik_nabiz_bpm,oda_sicakligi_celsius,hafta_sonu_uyku_farki_saat,mevsim,gun_tipi,bilissel_performans_skoru
0,1,34,Erkek,Saglik Personeli,31.470103,Cin,14.431210,14.645436,27,7,...,9.922976,10.045274,Sabah insani,Anksiyete ve depresyon,78,18.962436,-0.074140,Sonbahar-Kis,Hafta ici,0.136441
1,2,32,Kadin,Muhendis,30.981394,Amerika,21.771870,27.220360,20,4,...,6.626400,6.319245,Gece insani,Saglikli,76,21.225666,0.942672,Sonbahar-Kis,Hafta ici,5.848312
2,3,39,Erkek,Ev Hanimi,21.533898,Spain,18.178857,25.530104,33,7,...,6.093566,3.824463,Notr,Depresyon,66,18.482409,1.239886,Ilkbahar-Yaz,Hafta sonu,6.828276
3,4,40,Kadin,Egitimci,23.310749,Yeni Zelanda,21.438151,15.891188,21,2,...,3.168185,4.597316,Gece insani,Saglikli,60,21.862235,0.727695,Sonbahar-Kis,Hafta sonu,8.144649
4,5,36,Kadin,NaN,NaN,Portekiz,25.468018,16.356738,21,8,...,7.198574,3.189120,Notr,Anksiyete ve depresyon,74,19.223195,-0.223402,Sonbahar-Kis,Hafta ici,0.431423


,id,yas,cinsiyet,meslek,vucut_kitle_indeksi,ulke,rem_yuzdesi,derin_uyku_yuzdesi,uykuya_dalma_suresi_dk,gecelik_uyanma_sayisi,...,sekerleme_suresi_dk,stres_skoru,gunluk_calisma_saati,kronotip,ruh_sagligi_durumu,dinlenik_nabiz_bpm,oda_sicakligi_celsius,hafta_sonu_uyku_farki_saat,mevsim,gun_tipi
0,1,42,Erkek,Saglik Personeli,29.728724,Ingiltere,21.691917,23.990157,21,5,...,0,6.349196,9.419561,Gece insani,Depresyon,82,21.627129,1.716161,Ilkbahar-Yaz,Hafta sonu
1,2,26,Kadin,Serbest Calisan,32.865996,Cin,22.090624,18.231963,34,5,...,0,NaN,8.574199,Gece insani,Saglikli,66,27.835934,1.482283,Ilkbahar-Yaz,Hafta ici
2,3,21,Erkek,Lojistik Calisani,24.438264,Cin,19.488438,16.695363,17,4,...,52,6.715192,10.519017,Notr,Depresyon,63,20.252258,1.981034,Sonbahar-Kis,Hafta ici
3,4,61,Kadin,Saglik Personeli,23.275057,Cin,26.831092,21.762040,29,2,...,2,5.633400,9.814789,Gece insani,Saglikli,71,17.680971,0.589750,Sonbahar-Kis,Hafta ici
4,5,26,Kadin,Saglik Personeli,24.815531,Ingiltere,16.271522,18.153212,15,4,...,0,7.953794,9.308398,Sabah insani,Saglikli,53,23.698979,0.622211,Ilkbahar-Yaz,Hafta ici


,id,bilissel_performans_skoru
0,1,7.85
1,2,4.32


In [17]:
train_fe = add_features(train)
test_fe = add_features(test)

print("Train shape before FE:", train.shape)
print("Train shape after FE:", train_fe.shape)

print("Test shape before FE:", test.shape)
print("Test shape after FE:", test_fe.shape)

Train shape before FE: (56000, 24)
Train shape after FE: (56000, 35)
Test shape before FE: (24000, 23)
Test shape after FE: (24000, 34)


In [18]:
X = train_fe.drop(columns=[TARGET, ID_COL])
y = train_fe[TARGET]

X_test = test_fe.drop(columns=[ID_COL])
test_ids = test_fe[ID_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X shape: (56000, 33)
y shape: (56000,)
X_test shape: (24000, 33)


In [19]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric feature count:", len(numeric_features))
print("Categorical feature count:", len(categorical_features))

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric feature count: 25
Categorical feature count: 8

Numeric features:
['yas', 'vucut_kitle_indeksi', 'rem_yuzdesi', 'derin_uyku_yuzdesi', 'uykuya_dalma_suresi_dk', 'gecelik_uyanma_sayisi', 'uyku_oncesi_kafein_mg', 'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi', 'sekerleme_suresi_dk', 'stres_skoru', 'gunluk_calisma_saati', 'dinlenik_nabiz_bpm', 'oda_sicakligi_celsius', 'hafta_sonu_uyku_farki_saat', 'toplam_kaliteli_uyku_yuzdesi', 'rem_derin_uyku_carpim', 'uyku_bolunme_yuku', 'uyku_verimsizlik_skoru', 'stres_calisma_yuku', 'ekran_kafein_yuku', 'adim_sayisi_bin', 'nabiz_stres_yuku', 'hafta_sonu_flag', 'ruh_sagligi_risk_skoru']

Categorical features:
['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi', 'bmi_kategori']


In [20]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [21]:
def run_cv_model(model_name, model, X, y, X_test):
    print("=" * 80)
    print(f"Model: {model_name}")

    oof_pred = np.zeros(len(X))
    test_pred_folds = np.zeros((len(X_test), N_SPLITS))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        print(f"Fold {fold}")

        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        # Validation tahmini
        valid_pred = pipeline.predict(X_valid_fold)
        valid_pred = np.clip(valid_pred, 0, 10)

        fold_rmse = rmse(y_valid_fold, valid_pred)
        fold_scores.append(fold_rmse)

        oof_pred[valid_idx] = valid_pred

        # Gerçek Kaggle test verisi için tahmin
        test_pred = pipeline.predict(X_test)
        test_pred = np.clip(test_pred, 0, 10)

        test_pred_folds[:, fold - 1] = test_pred

        print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

    mean_rmse = np.mean(fold_scores)
    std_rmse = np.std(fold_scores)

    print(f"{model_name} CV RMSE: {mean_rmse:.5f} ± {std_rmse:.5f}")

    return {
        "model": model_name,
        "cv_rmse_mean": mean_rmse,
        "cv_rmse_std": std_rmse,
        "fold_scores": fold_scores,
        "oof_pred": oof_pred,
        "test_pred": test_pred_folds.mean(axis=1)
    }

In [22]:
catboost_experiments = {
    "catboost_base": CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=3,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),

    "catboost_depth4_lr003": CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=4,
        l2_leaf_reg=3,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),

    "catboost_depth5_lr003": CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=3,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),

    "catboost_depth6_lr002": CatBoostRegressor(
        iterations=4000,
        learning_rate=0.02,
        depth=6,
        l2_leaf_reg=3,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),

    "catboost_depth6_l2_5": CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),

    "catboost_depth7_lr002": CatBoostRegressor(
        iterations=4000,
        learning_rate=0.02,
        depth=7,
        l2_leaf_reg=5,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),
}

In [23]:
all_results = []
all_oof_predictions = {}
all_test_predictions = {}

for model_name, model in catboost_experiments.items():
    result = run_cv_model(model_name, model, X, y, X_test)

    all_results.append({
        "model": result["model"],
        "cv_rmse_mean": result["cv_rmse_mean"],
        "cv_rmse_std": result["cv_rmse_std"],
        "fold_scores": result["fold_scores"]
    })

    all_oof_predictions[model_name] = result["oof_pred"]
    all_test_predictions[model_name] = result["test_pred"]

Model: catboost_base
Fold 1
Fold 1 RMSE: 1.22419
Fold 2
Fold 2 RMSE: 1.22403
Fold 3
Fold 3 RMSE: 1.20821
Fold 4
Fold 4 RMSE: 1.21970
Fold 5
Fold 5 RMSE: 1.23827
catboost_base CV RMSE: 1.22288 ± 0.00965
Model: catboost_depth4_lr003
Fold 1
Fold 1 RMSE: 1.22045
Fold 2
Fold 2 RMSE: 1.22000
Fold 3
Fold 3 RMSE: 1.20665
Fold 4
Fold 4 RMSE: 1.21403
Fold 5
Fold 5 RMSE: 1.23523
catboost_depth4_lr003 CV RMSE: 1.21927 ± 0.00942
Model: catboost_depth5_lr003
Fold 1
Fold 1 RMSE: 1.22317
Fold 2
Fold 2 RMSE: 1.22156
Fold 3
Fold 3 RMSE: 1.20780
Fold 4
Fold 4 RMSE: 1.21690
Fold 5
Fold 5 RMSE: 1.23816
catboost_depth5_lr003 CV RMSE: 1.22152 ± 0.00989
Model: catboost_depth6_lr002
Fold 1
Fold 1 RMSE: 1.22288
Fold 2
Fold 2 RMSE: 1.22167
Fold 3
Fold 3 RMSE: 1.20699
Fold 4
Fold 4 RMSE: 1.21783
Fold 5
Fold 5 RMSE: 1.23700
catboost_depth6_lr002 CV RMSE: 1.22127 ± 0.00966
Model: catboost_depth6_l2_5
Fold 1
Fold 1 RMSE: 1.22420
Fold 2
Fold 2 RMSE: 1.22426
Fold 3
Fold 3 RMSE: 1.20810
Fold 4
Fold 4 RMSE: 1.21807
Fold

In [24]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("cv_rmse_mean").reset_index(drop=True)

results_df[["model", "cv_rmse_mean", "cv_rmse_std"]]

,model,cv_rmse_mean,cv_rmse_std
0,catboost_depth4_lr003,1.219271,0.009419
1,catboost_depth6_lr002,1.221273,0.009655
2,catboost_depth5_lr003,1.221517,0.009891
3,catboost_depth6_l2_5,1.222680,0.009974
4,catboost_base,1.222878,0.009646
5,catboost_depth7_lr002,1.223139,0.009889


In [25]:
best_model_name = results_df.loc[0, "model"]
best_test_pred = all_test_predictions[best_model_name]

best_test_pred = np.clip(best_test_pred, 0, 10)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_test_pred
})

print("Best model:", best_model_name)
print("Submission shape:", submission.shape)
print("Submission columns:", submission.columns.tolist())

display(submission.head())

Best model: catboost_depth4_lr003
Submission shape: (24000, 2)
Submission columns: ['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,6.037308
1,2,6.353497
2,3,2.918269
3,4,7.176370
4,5,3.650912


In [26]:
print("Our submission shape:", submission.shape)
print("Sample submission shape:", sample_submission.shape)

print("\nOur columns:")
print(submission.columns.tolist())

print("\nSample columns:")
print(sample_submission.columns.tolist())

display(submission.head())
display(sample_submission.head())

Our submission shape: (24000, 2)
Sample submission shape: (2, 2)

Our columns:
['id', 'bilissel_performans_skoru']

Sample columns:
['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,6.037308
1,2,6.353497
2,3,2.918269
3,4,7.176370
4,5,3.650912


,id,bilissel_performans_skoru
0,1,7.85
1,2,4.32


In [27]:
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

submission_path = SUBMISSION_DIR / f"submission_{best_model_name}.csv"

submission.to_csv(submission_path, index=False)

print("Best model:", best_model_name)
print("Saved:", submission_path)
print("Shape:", submission.shape)

Best model: catboost_depth4_lr003
Saved: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_catboost_depth4_lr003.csv
Shape: (24000, 2)


In [28]:
check_submission = pd.read_csv(submission_path)

print("Check shape:", check_submission.shape)
print("Check columns:", check_submission.columns.tolist())

display(check_submission.head())

Check shape: (24000, 2)
Check columns: ['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,6.037308
1,2,6.353497
2,3,2.918269
3,4,7.176370
4,5,3.650912


In [29]:
PREDICTION_DIR = PROJECT_ROOT / "predictions" / "catboost"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

for model_name in all_oof_predictions.keys():
    oof_df = pd.DataFrame({
        ID_COL: train[ID_COL],
        "y_true": y,
        "oof_pred": np.clip(all_oof_predictions[model_name], 0, 10)
    })

    test_df = pd.DataFrame({
        ID_COL: test_ids,
        "test_pred": np.clip(all_test_predictions[model_name], 0, 10)
    })

    oof_path = PREDICTION_DIR / f"{model_name}_oof.csv"
    test_path = PREDICTION_DIR / f"{model_name}_test.csv"

    oof_df.to_csv(oof_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Saved:", oof_path)
    print("Saved:", test_path)

Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_base_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_base_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_depth4_lr003_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_depth4_lr003_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_depth5_lr003_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_depth5_lr003_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_depth6_lr002_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_depth6_lr002_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_depth6_l2_5_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost\catboost_depth6_l2_5_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datath